In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_GAMES = "grandmasters.silver.game_events"

DEVELOPMENT_MODE = True

In [0]:
# ============================================================
# READ BRONZE
# ============================================================

bronze_df = spark.table(BRONZE_TABLE)

display(bronze_df.limit(10))

In [0]:
print("Total Bronze Records:", bronze_df.count())

bronze_df.groupBy("event_type").count().show()

bronze_df.select("game_id").distinct().show()

In [0]:
# ============================================================
# EXTRACT GAME EVENTS
#
# Purpose:
# Parse 'featured' events from Bronze into structured columns.
# One record represents one chess game.
# ============================================================

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

featured_schema = StructType([
    StructField("t", StringType()),
    StructField("d", StructType([
        StructField("id", StringType()),
        StructField("orientation", StringType()),
        StructField("fen", StringType()),
        StructField("winner", StringType()),
        StructField("players", ArrayType(
            StructType([
                StructField("color", StringType()),
                StructField("rating", IntegerType()),
                StructField("seconds", IntegerType()),
                StructField("user", StructType([
                    StructField("name", StringType()),
                    StructField("id", StringType())
                ]))
            ])
        ))
    ]))
])

game_events_df = (
    bronze_df
    .filter(col("event_type") == "featured")
    .withColumn(
        "json",
        from_json(col("raw_event"), featured_schema)
    )
    .select(
        col("game_id"),
        col("ingested_at"),

        col("json.d.players")[0]["user"]["name"].alias("white_player"),
        col("json.d.players")[1]["user"]["name"].alias("black_player"),

        col("json.d.players")[0]["rating"].alias("white_rating"),
        col("json.d.players")[1]["rating"].alias("black_rating"),

        col("json.d.players")[0]["seconds"].alias("time_control_seconds"),

        col("json.d.winner").alias("winner"),

        col("json.d.fen").alias("starting_fen")
    )
)

display(game_events_df)

In [0]:
# ============================================================
# VALIDATE GAME EVENTS
# ============================================================

from pyspark.sql.functions import countDistinct, col, sum as Fsum, col as Fcol

print("Total Games")
print(game_events_df.count())

print("\nDistinct Game IDs")
print(game_events_df.select(countDistinct("game_id")).first()[0])

print("\nDuplicate Games")

display(
    game_events_df
        .groupBy("game_id")
        .count()
        .filter("count > 1")
)

print("\nMissing Values")

game_events_df.select(
    *[
        Fsum(Fcol(c).isNull().cast("int")).alias(c)
        for c in game_events_df.columns
    ]
).display()

In [0]:
# ============================================================
# GAME EVENTS DATA QUALITY
#
# Business Rule:
# Keep only the latest record for each game.
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

latest_game_window = (
    Window
    .partitionBy("game_id")
    .orderBy(desc("ingested_at"))
)

game_events_df = (
    game_events_df
    .withColumn("rn", row_number().over(latest_game_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
# ============================================================
# WRITE SILVER GAME EVENTS
# ============================================================

(
    game_events_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_GAMES)
)

print(f"Successfully wrote {game_events_df.count()} rows to {SILVER_GAMES}")

In [0]:
display(spark.table(SILVER_GAMES))

In [0]:
spark.table(SILVER_GAMES).printSchema()